# Main Code

In [ ]:
from sklearn.cluster import KMeans
import hdbscan
import numpy as np

def get_K_Means(X,true_k):
    kmeans = KMeans(n_clusters=true_k,  max_iter=1000)
    kmeans.fit(X)
    labels_km = kmeans.labels_
    return labels_km

def get_HDBSCAN(X,true_k):
    hdb = hdbscan.HDBSCAN(prediction_data=True)
    hdb.fit(X)
    membership_probs = hdbscan.all_points_membership_vectors(hdb)
    if membership_probs.ndim == 1:
        labels_hdb = np.zeros_like(membership_probs, dtype=int)
    else:
        labels_hdb = np.argmax(membership_probs, axis=1)
    return labels_hdb

In [ ]:
# import libraries 
import numpy as np
import cs_kmeans as cs_km
import cs_hdbscan as cs_hdb
from sklearn.metrics.cluster import normalized_mutual_info_score as NMI
from sklearn.metrics.cluster import adjusted_rand_score as ARI

import warnings
warnings.filterwarnings("ignore", category=RuntimeWarning)
warnings.filterwarnings("ignore", category=FutureWarning, message=".force_all_finite.")


# Load Data
directory = "./"

def CORESPECT(d_name):
    # Load Data
    X = np.load(directory + 'dataset/'+d_name+'_X.npy', allow_pickle=True)
    label = np.load(directory + 'dataset/'+d_name+'_label.npy', allow_pickle=True)
    true_k = len(set(label))

    print("Number of clusters: ", true_k, " Number of samples: ", X.shape[0], " Number of features: ", X.shape[1])

    print(f"K-Means")
    pred_label= get_K_Means(X,true_k)
    print(f'NMI: {NMI(label, pred_label):.3f}, ARI: {ARI(label, pred_label):.3f}')


    # Run CoreSPECTed-K-Means
    layer_ratio = [0.1,0.2,0.3,0.4,0.5,0.6,0.7,0.8,0.9,1]
    pred_label = cs_km.MCPC_Kmeans(X, true_k, q=40, r=20, t=20, layer_ratio=layer_ratio)
    print(f"CoreSPECTed-K-Means")
    print(f'NMI: {NMI(label, pred_label):.3f}, ARI: {ARI(label, pred_label):.3f}')


    #Run HDBSCAN
    print(f"HDBSCAN")
    pred_label= get_HDBSCAN(X,true_k)
    print(f'NMI: {NMI(label, pred_label):.3f}, ARI: {ARI(label, pred_label):.3f}')


    # Run CoreSPECTed-HDBSCAN
    layer_ratio = [0.1,0.2,0.3,0.4,0.5,0.6,0.7,0.8,0.9,1]
    pred_label = cs_hdb.MCPC_HDBSCAN(X, true_k, q=40, r=20, t=20, layer_ratio=layer_ratio)
    print(f"CoreSPECTed-HDBSCAN")
    print(f'NMI: {NMI(label, pred_label):.3f}, ARI: {ARI(label, pred_label):.3f}')

In [ ]:
data_names=['miRNA','AMB','Xin']

for d_name in data_names:
    print(f'Running CoreSPECT on {d_name} dataset')
    CORESPECT(d_name)
    print('----------------------------------')